# Phase 8 one-time reserved evaluation

Run every cell once, in order. This notebook evaluates the frozen 2026 reserved period and refuses to overwrite an existing result. Do not change features, cutoffs, dates, metrics, or procedures after results appear.

In [ ]:
from pathlib import Path
import os, subprocess, sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'https://github.com/frager098/HalalifyAI.git'
REPO_REF = 'feature/model-evaluation'
if IN_COLAB:
    REPO_ROOT = Path('/content/HalalifyAI')
    if not (REPO_ROOT / 'AGENTS.md').is_file():
        subprocess.run(['git', 'clone', '--branch', REPO_REF, '--single-branch', REPO_URL, str(REPO_ROOT)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', REPO_REF], check=True)
else:
    start = Path.cwd().resolve()
    candidates = [start, *start.parents]
    REPO_ROOT = next((p for p in candidates if (p / 'AGENTS.md').is_file() and (p / 'src').is_dir() and (p / 'configs').is_dir()), None)
    if REPO_ROOT is None:
        raise FileNotFoundError('Could not locate the HalalifyAI repository root.')

os.chdir(REPO_ROOT)
print('Environment:', 'Google Colab' if IN_COLAB else 'Local Python')
print('Repository root:', REPO_ROOT)
print('Expected Git ref:', REPO_REF)

In [ ]:
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-cleaning.txt'], check=True)
print('Dependencies ready.')

In [ ]:
required_code = [
    Path('configs/phase8_evaluation_contract_v1.json'),
    Path('configs/phase8_procedure_freeze_v1.json'),
    Path('src/model_training/phase8/evaluate_once.py'),
    Path('src/model_training/phase8/runner.py'),
]
missing_code = [str(path) for path in required_code if not path.is_file()]
if missing_code:
    raise FileNotFoundError('The cloned branch lacks required Phase 8 code. Commit and push feature/model-evaluation first. Missing: ' + ', '.join(missing_code))

required_uploads = {
    'reserved_features_and_labels.csv': Path('data/processed/phase8_model_inputs/phase8_reserved_evaluation_v1/reserved_features_and_labels.csv'),
    'dataset_metadata.json': Path('data/processed/phase8_model_inputs/phase8_reserved_evaluation_v1/dataset_metadata.json'),
    'validation_report.json': Path('data/processed/phase8_model_inputs/phase8_reserved_evaluation_v1/validation_report.json'),
    'runner_validation.json': Path('reports/model_evaluation/phase8_reserved_evaluation_v1/runner_validation.json'),
}
missing = {name: path for name, path in required_uploads.items() if not path.is_file()}
if IN_COLAB and missing:
    from google.colab import files
    print('Upload these files:', ', '.join(missing))
    uploaded = files.upload()
    for name, destination in missing.items():
        if name not in uploaded:
            raise FileNotFoundError(f'Missing upload: {name} -> {destination}')
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(uploaded[name])
missing_after = [str(path) for path in required_uploads.values() if not path.is_file()]
if missing_after:
    raise FileNotFoundError('Required inputs are missing at: ' + ', '.join(missing_after))
print('Preflight passed. Required runtime files are present.')

In [ ]:
from src.model_training.phase8.procedure import validate_runtime_gate
gate = validate_runtime_gate()
assert gate['ready_for_one_time_evaluation'] is True
print('Frozen runtime gate passed:', gate)

In [ ]:
from src.model_training.phase8.evaluate_once import run_one_time_evaluation
result = run_one_time_evaluation()
print('One-time evaluation written successfully.')
print('Return mean daily Spearman:', result['return']['mean_daily_spearman'])
print('Risk macro F1:', result['risk']['macro_f1'])
print('Result path: reports/model_evaluation/phase8_reserved_evaluation_v1/one_time_evaluation.json')